# Mutation-impact molecular dynamics workflow

This notebook performs replica-based structural ensemble analysis for a validated soluble-protein system. Membrane, ligand-bound, cofactor-dependent, metal-containing, and complex systems require their dedicated builders and parameterization branches; they are blocked from the generic solvent workflow rather than silently simplified.


In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import time

import MDAnalysis as mda
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from MDAnalysis.analysis.rms import RMSD, RMSF
from openmm.app import PDBFile
from pdbfixer import PDBFixer

PDB_FILE = "uploads/ppox/3NKS_WT.pdb"
PROJECT_PREFIX = "PPOX_Study"
SYSTEM_TYPE = "soluble"  # soluble, membrane, complex, ligand_bound, metal, disordered
BIOLOGICAL_ASSEMBLY_FILE = None
OLIGOMER_CHAINS = []
TARGET_CHAIN_ID = "A"
MUTATION_STR = "ALA-449-THR"
MUTATION_RES_ID = 449
MUTATION_FROM = "ALA"
MUTATION_TO = "THR"
APPLY_MUTATION = True
FORCE_FIELD = "charmm36m"
WATER_MODEL = "tip3p"
BOX_TYPE = "dodecahedron"
BOX_BUFFER_NM = 1.0
TEMPERATURE = 300.0
PRESSURE_BAR = 1.0
SALT_CONCENTRATION_M = 0.15
PH = 7.0
PROTONATION_METHOD = "manual_review"
SIMULATION_TIME_NS = 100
N_REPLICAS = 3
GPU_ID = "0"
ANALYSIS_ENDPOINT = "structural_ensemble"
ANALYSIS_START_NS = max(0.0, SIMULATION_TIME_NS - 200.0)
CONTACT_CUTOFF_A = 5.0
RUN_INTERFACE_ANALYSIS = bool(OLIGOMER_CHAINS)
RUN_MMGBSA = False
RUN_ALCHEMICAL_FREE_ENERGY = False
KEEP_WATER = True
REMOVE_HETEROGENS = False
LIGAND_RESNAMES = []
COFACTOR_RESNAMES = []
METAL_RESNAMES = []

if SYSTEM_TYPE != "soluble":
    raise ValueError("The generic branch is only for soluble proteins. Use CHARMM-GUI or a validated component-specific workflow for this system type.")
if BIOLOGICAL_ASSEMBLY_FILE is None and OLIGOMER_CHAINS:
    raise ValueError("Provide a biological assembly structure before declaring oligomer chains.")
if REMOVE_HETEROGENS and (LIGAND_RESNAMES or COFACTOR_RESNAMES or METAL_RESNAMES):
    raise ValueError("Required ligands, cofactors, or metals cannot be removed.")
if N_REPLICAS < 3:
    raise ValueError("At least three independent replicas are required for comparative analysis.")
if ANALYSIS_ENDPOINT in {"folding_ddg", "binding_ddg"} and not RUN_ALCHEMICAL_FREE_ENERGY:
    raise ValueError("Folding/binding ddG requires an explicit alchemical free-energy workflow.")
if RUN_MMGBSA:
    raise NotImplementedError("MM/GBSA setup is not implemented yet; keep RUN_MMGBSA=False until a validated interface workflow is added.")

NOTEBOOK_ROOT = Path.cwd().resolve()
PDB_PATH = (NOTEBOOK_ROOT / (BIOLOGICAL_ASSEMBLY_FILE or PDB_FILE)).resolve()
WORK_DIR_BASE = NOTEBOOK_ROOT / "results"
SYSTEMS = ["WT", "Mutant"] if APPLY_MUTATION else ["WT"]
REPLICAS = [f"replica_{i:02d}" for i in range(1, N_REPLICAS + 1)]

print(f"Configured {SYSTEM_TYPE} system: {SYSTEMS}, {N_REPLICAS} replicas each")
print(f"Analysis window starts at {ANALYSIS_START_NS:.1f} ns; contact cutoff: {CONTACT_CUTOFF_A:.1f} A")

Configured soluble system: ['WT', 'Mutant'], 3 replicas each
Analysis window starts at 0.0 ns; contact cutoff: 5.0 A


## 1. Structure preparation and mutation validation

Non-protein components are retained by default. Protonation, ligand, cofactor, and metal parameters must be reviewed before proceeding.


In [ ]:
def run_gmx(command, cwd, stdin_text=None):
    result = subprocess.run(command, cwd=cwd, input=stdin_text, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        raise RuntimeError(f"GROMACS failed: {' '.join(command)}")
    return result.stdout

def patch_mdp(path, updates):
    text = path.read_text()
    for key, value in updates.items():
        text, count = re.subn(rf"^{re.escape(key)}\s*=.*$", f"{key} = {value}", text, flags=re.MULTILINE)
        if count != 1:
            raise ValueError(f"Expected one {key} entry in {path}, found {count}")
    path.write_text(text)

def read_xvg(path):
    rows = []
    for line in path.read_text().splitlines():
        if line and not line.startswith(("@", "#")):
            rows.append([float(value) for value in line.split()])
    return pd.DataFrame(rows)

for system in SYSTEMS:
    output_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}"
    output_dir.mkdir(parents=True, exist_ok=True)
    fixer = PDBFixer(filename=str(PDB_PATH))
    fixer.findMissingResidues()
    if fixer.missingResidues:
        missing = {key: len(value) for key, value in fixer.missingResidues.items() if value}
        raise ValueError(f"Unresolved residues detected: {missing}. Model and validate these loops before MD.")
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    if REMOVE_HETEROGENS:
        fixer.removeHeterogens(keepWater=KEEP_WATER)
    if system == "Mutant":
        chains = [chain.id for chain in fixer.topology.chains()]
        if TARGET_CHAIN_ID not in chains:
            raise ValueError(f"Missing target chain {TARGET_CHAIN_ID}; available: {chains}")
        matches = [residue for chain in fixer.topology.chains() if chain.id == TARGET_CHAIN_ID for residue in chain.residues() if residue.id == str(MUTATION_RES_ID)]
        if len(matches) != 1 or matches[0].name != MUTATION_FROM:
            raise ValueError(f"Expected {MUTATION_FROM} at {TARGET_CHAIN_ID}:{MUTATION_RES_ID}")
        fixer.applyMutations([MUTATION_STR], TARGET_CHAIN_ID)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(PH)
    with open(output_dir / "prepared.pdb", "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle)
    print(f"Prepared {system}: {output_dir / 'prepared.pdb'}")

Prepared WT: /home/kmu/Taimoor/Genomics/research_bioinformatics/md-simulation/results/PPOX_Study_WT/prepared.pdb
Prepared Mutant: /home/kmu/Taimoor/Genomics/research_bioinformatics/md-simulation/results/PPOX_Study_Mutant/prepared.pdb


## 2. System building, equilibration, and independent production replicas

This branch uses matched CHARMM36m/TIP3P solvent, does not use `-maxwarn`, and uses named GROMACS groups rather than numeric selections. Membrane systems must use a membrane builder with semi-isotropic pressure coupling.


In [ ]:
for system in SYSTEMS:
    source_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}"
    for replica in REPLICAS:
        run_dir = source_dir / replica
        run_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "prepared.pdb", run_dir / "prepared.pdb")
        shutil.copytree(NOTEBOOK_ROOT / "mdp_templates", run_dir / "mdp", dirs_exist_ok=True)
        run_gmx(["gmx", "pdb2gmx", "-f", "prepared.pdb", "-o", "protein.gro", "-p", "topol.top", "-i", "posre.itp", "-ff", FORCE_FIELD, "-water", WATER_MODEL, "-ignh", "-ter"], run_dir, "1\n0\n")
        run_gmx(["gmx", "editconf", "-f", "protein.gro", "-o", "box.gro", "-c", "-d", str(BOX_BUFFER_NM), "-bt", BOX_TYPE], run_dir)
        run_gmx(["gmx", "solvate", "-cp", "box.gro", "-cs", "tip3p.gro", "-o", "solvated.gro", "-p", "topol.top"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/em.mdp", "-c", "solvated.gro", "-r", "solvated.gro", "-p", "topol.top", "-o", "ions.tpr"], run_dir)
        run_gmx(["gmx", "genion", "-s", "ions.tpr", "-o", "solv_ions.gro", "-p", "topol.top", "-pname", "NA", "-nname", "CL", "-neutral", "-conc", str(SALT_CONCENTRATION_M)], run_dir, "SOL\n")

        patch_mdp(run_dir / "mdp/nvt.mdp", {"ref_t": f"{TEMPERATURE} {TEMPERATURE}"})
        patch_mdp(run_dir / "mdp/npt.mdp", {"ref_t": f"{TEMPERATURE} {TEMPERATURE}", "ref_p": str(PRESSURE_BAR), "refcoord-scaling": "all"})
        run_gmx(["gmx", "grompp", "-f", "mdp/em.mdp", "-c", "solv_ions.gro", "-r", "solv_ions.gro", "-p", "topol.top", "-o", "em.tpr"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "em", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/nvt.mdp", "-c", "em.gro", "-r", "em.gro", "-p", "topol.top", "-o", "nvt.tpr"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "nvt", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/npt.mdp", "-c", "nvt.gro", "-r", "nvt.gro", "-p", "topol.top", "-o", "npt.tpr", "-t", "nvt.cpt"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "npt", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)

        prod_steps = int(SIMULATION_TIME_NS * 1000 / 0.002)
        patch_mdp(run_dir / "mdp/md.mdp", {"nsteps": prod_steps, "ref_t": str(TEMPERATURE), "tc-grps": "System", "tau_t": "1.0", "pcoupl": "Parrinello-Rahman", "tau_p": "5.0", "refcoord-scaling": "all"})
        run_gmx(["gmx", "grompp", "-f", "mdp/md.mdp", "-c", "npt.gro", "-r", "npt.gro", "-p", "topol.top", "-o", "md.tpr", "-t", "npt.cpt"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "md", "-gpu_id", GPU_ID, "-nb", "gpu", "-pme", "gpu", "-bonded", "gpu", "-update", "gpu"], run_dir)
        run_gmx(["gmx", "trjconv", "-s", "md.tpr", "-f", "md.xtc", "-o", "md_center.xtc", "-pbc", "mol", "-center"], run_dir, "Protein\nSystem\n")
        run_gmx(["gmx", "trjconv", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "md_fit.xtc", "-fit", "rot+trans"], run_dir, "Backbone\nSystem\n")
        print(f"Completed {system}/{replica}")


## 3. Replica-aware analysis and interpretation gate

The analysis reports structural ensemble descriptors with replica-level variability: RMSD, RMSF, solvent-accessible surface area (SASA), radius of gyration, and mutation-neighborhood contact occupancy. Contact analysis is enabled only when a validated biological assembly and oligomer-chain configuration are provided.

The default 100 ns run is exploratory. Set `SIMULATION_TIME_NS = 1000` for a paper-comparable 1 microsecond production campaign after a short validation run has completed successfully. RMSD, RMSF, SASA, radius of gyration, and contact occupancy are not folding or binding free energies.

In [ ]:
def residue_center_positions(residues):
    return pd.DataFrame([residue.atoms.center_of_mass() for residue in residues]).to_numpy()

summary_records = []
contact_records = []

for system in SYSTEMS:
    for replica in REPLICAS:
        run_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}" / replica
        run_gmx(["gmx", "sasa", "-s", "md.tpr", "-f", "md_fit.xtc", "-o", "sasa.xvg"], run_dir, "Protein\n")
        run_gmx(["gmx", "gyrate", "-s", "md.tpr", "-f", "md_fit.xtc", "-o", "gyrate.xvg"], run_dir, "Protein\n")

        universe = mda.Universe(str(run_dir / "md.gro"), str(run_dir / "md_fit.xtc"))
        backbone = universe.select_atoms("protein and backbone")
        protein = universe.select_atoms("protein")
        residues = list(protein.residues)
        residue_ids = [int(residue.resid) for residue in residues]
        rmsd = RMSD(universe, select="protein and backbone").run().results.rmsd
        rmsf = RMSF(backbone).run().results.rmsf / 10.0
        analysis_frame_start = SIMULATION_TIME_NS * 1000.0 - ANALYSIS_START_NS * 1000.0
        tail_mask = rmsd[:, 1] >= ANALYSIS_START_NS * 1000.0
        tail_rmsd = rmsd[tail_mask, 2].mean() / 10.0
        local_mask = backbone.resids >= MUTATION_RES_ID - 5
        local_mask &= backbone.resids <= MUTATION_RES_ID + 5
        sasa = read_xvg(run_dir / "sasa.xvg")
        gyrate = read_xvg(run_dir / "gyrate.xvg")
        sasa_tail = sasa[sasa[0] >= ANALYSIS_START_NS * 1000.0][1].mean()
        rg_tail = gyrate[gyrate[0] >= ANALYSIS_START_NS * 1000.0][1].mean()
        summary_records.append({
            "System": system,
            "Replica": replica,
            "tail_rmsd_nm": tail_rmsd,
            "global_rmsf_nm": rmsf.mean(),
            "mutation_window_rmsf_nm": rmsf[local_mask].mean(),
            "tail_sasa_nm2": sasa_tail,
            "tail_rg_nm": rg_tail,
        })

        if RUN_INTERFACE_ANALYSIS:
            candidate_indices = [index for index, resid in enumerate(residue_ids) if abs(resid - MUTATION_RES_ID) <= 5]
            all_indices = list(range(len(residues)))
            contact_counts = {}
            analyzed_frames = 0
            for timestep in universe.trajectory:
                if timestep.time < ANALYSIS_START_NS * 1000.0:
                    continue
                centers = residue_center_positions(residues)
                analyzed_frames += 1
                for index_i in candidate_indices:
                    distances = ((centers[index_i] - centers) ** 2).sum(axis=1) ** 0.5
                    for index_j in all_indices:
                        if index_i == index_j or distances[index_j] * 10.0 >= CONTACT_CUTOFF_A:
                            continue
                        pair = tuple(sorted((residue_ids[index_i], residue_ids[index_j])))
                        contact_counts[pair] = contact_counts.get(pair, 0) + 1
            if analyzed_frames == 0:
                raise ValueError(f"No frames available after {ANALYSIS_START_NS} ns in {run_dir}")
            for (resid_i, resid_j), count in contact_counts.items():
                contact_records.append({
                    "System": system,
                    "Replica": replica,
                    "resid_i": resid_i,
                    "resid_j": resid_j,
                    "occupancy": count / analyzed_frames,
                })

summary = pd.DataFrame(summary_records)
if summary.groupby("System")["Replica"].nunique().min() < N_REPLICAS:
    raise ValueError("Not all requested replicas completed.")
summary_stats = summary.groupby("System").agg(["mean", "std", "count"])
display(summary_stats)
summary.to_csv(WORK_DIR_BASE / f"{PROJECT_PREFIX}_replica_summary.csv", index=False)

if contact_records:
    contacts = pd.DataFrame(contact_records)
    contact_stats = contacts.groupby(["System", "resid_i", "resid_j"])["occupancy"].agg(["mean", "std", "count"]).reset_index()
    contact_pivot = contact_stats.pivot_table(index=["resid_i", "resid_j"], columns="System", values="mean", fill_value=0.0)
    if {"WT", "Mutant"}.issubset(contact_pivot.columns):
        contact_pivot["delta_occupancy"] = contact_pivot["Mutant"] - contact_pivot["WT"]
    contacts.to_csv(WORK_DIR_BASE / f"{PROJECT_PREFIX}_contact_occupancy_by_replica.csv", index=False)
    contact_stats.to_csv(WORK_DIR_BASE / f"{PROJECT_PREFIX}_contact_occupancy_summary.csv", index=False)
    display(contact_pivot.sort_values("delta_occupancy", ascending=False).head(20))

sns.set_theme(style="ticks", context="paper", font_scale=1.1)
fig, axes = plt.subplots(2, 3, figsize=(18, 9), dpi=200)
for metric, axis, title in [
    ("tail_rmsd_nm", axes[0, 0], "Tail backbone RMSD"),
    ("global_rmsf_nm", axes[0, 1], "Global backbone RMSF"),
    ("mutation_window_rmsf_nm", axes[0, 2], "Mutation-window RMSF"),
    ("tail_sasa_nm2", axes[1, 0], "Tail SASA"),
    ("tail_rg_nm", axes[1, 1], "Tail radius of gyration"),
]:
    sns.stripplot(data=summary, x="System", y=metric, hue="System", legend=False, ax=axis)
    sns.pointplot(data=summary, x="System", y=metric, errorbar="sd", color="black", ax=axis)
    axis.set_title(title)
axes[1, 2].axis("off")
sns.despine()
plt.tight_layout()
plt.savefig(WORK_DIR_BASE / f"{PROJECT_PREFIX}_Comparative_Analysis.png", dpi=300)
plt.show()
print("Replica-aware structural and contact analysis completed.")

## Methodological boundary

For folding stability, calculate a mutation free-energy difference using a validated alchemical thermodynamic cycle, such as pmx/FEP/TI with lambda-window overlap and independent convergence checks. For ligand binding, include both apo and complex legs. MM/GBSA may be added as an approximate comparative screening method for a validated oligomeric interface, but it must not be reported as a rigorous binding free energy.

For membranes, ligands, cofactors, metals, or partner complexes, construct the biological system and parameterize every retained component before production. The generic soluble-protein branch intentionally stops for these cases.